In [10]:
import numpy as np
import pandas as pd

In [11]:
job_df=pd.read_csv("data/job_info_df.csv")
node_df=pd.read_csv("data/node_info_df.csv")

In [ ]:
print(node_df.shape)
print(node_df.head())

In [14]:
print(job_df.shape)
print(job_df.head())

(466867, 9)
   job_name  organization     gpu_model  cpu_request  gpu_request  worker_num  \
0    239255            13           A10         20.0          1.0           1   
1    253689            13           A10          8.0          1.0           1   
2    236907            13           A10          8.0          1.0           1   
3    253671            13  GPU-series-1          4.0          1.0           1   
4    236901            13           A10          8.0          1.0           1   

   submit_time    duration job_type  
0          0.0   2764799.0       HP  
1          0.0  15897599.0       HP  
2          0.0   9857174.0       HP  
3          0.0   1727999.0       HP  
4          0.0  15897599.0       HP  


In [4]:
print(job_df['job_type'].value_counts(normalize=True)*100)


job_type
HP      89.043132
Spot    10.956868
Name: proportion, dtype: float64


In [5]:
print(job_df['gpu_model'].value_counts())

gpu_model
A10               228643
A100-SXM4-80GB    169020
GPU-series-2       30397
GPU-series-1       18911
H800               11097
A800-SXM4-80GB      8799
Name: count, dtype: int64


In [ ]:
print(job_df['duration'].describe())
print(job_df['worker_num'].describe())

In [ ]:
hp_jobs=job_df[job_df['job_type']=='HP'].copy()
hp_jobs

In [ ]:
spot_jobs=job_df[job_df['job_type']=='Spot'].copy()
spot_jobs

In [17]:
print(f"HP jobs: {len(hp_jobs)}")
print(f"Spot jobs: {len(spot_jobs)}")

HP jobs: 415713
Spot jobs: 51154


In [24]:
print("Duration per jobs")
print(hp_jobs['duration'].describe())
print(spot_jobs['duration'].describe())

Duration per jobs
count    4.157130e+05
mean     2.065210e+05
std      1.122888e+06
min      1.000000e+00
25%      2.050000e+02
50%      1.101000e+03
75%      4.009000e+03
max      1.589760e+07
Name: duration, dtype: float64
count    5.115400e+04
mean     1.367958e+04
std      8.455115e+04
min      1.000000e+00
25%      5.430000e+02
50%      2.095500e+03
75%      8.630000e+03
max      5.848226e+06
Name: duration, dtype: float64


In [23]:
print("Worker numbers per jobs")
print(hp_jobs['worker_num'].describe())
print(spot_jobs['worker_num'].describe())



Worker numbers per jobs
count    415713.000000
mean          1.383568
std           3.643022
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         182.000000
Name: worker_num, dtype: float64
count    51154.000000
mean         4.186476
std         12.972041
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        332.000000
Name: worker_num, dtype: float64


In [ ]:
#End Time for every jobs
job_df['end_time']=job_df['submit_time']+job_df['duration']
job_df


In [26]:
hp_jobs=job_df[job_df['job_type']=="HP"].copy()
spot_jobs=job_df[job_df['job_type']=="Spot"].copy()

In [28]:
spot_jobs

,job_name,organization,gpu_model,cpu_request,gpu_request,worker_num,submit_time,duration,job_type,end_time
3839,415713,57,A100-SXM4-80GB,112.0,8.0,1,4959.0,2496.0,Spot,7455.0
27150,415714,43,H800,10.0,4.0,1,401311.0,167.0,Spot,401478.0
36596,415715,43,A100-SXM4-80GB,20.0,4.0,1,649979.0,4290.0,Spot,654269.0
69288,415716,77,A800-SXM4-80GB,16.0,1.0,2,1867576.0,441.0,Spot,1868017.0
81454,415717,78,A100-SXM4-80GB,16.0,1.0,1,2374656.0,2538.0,Spot,2377194.0
...,...,...,...,...,...,...,...,...,...,...
466818,466862,57,GPU-series-2,10.0,1.0,44,15894063.0,471.0,Spot,15894534.0
466819,466864,57,GPU-series-2,10.0,1.0,17,15894064.0,470.0,Spot,15894534.0
466820,466863,57,GPU-series-1,10.0,1.0,5,15894064.0,466.0,Spot,15894530.0
466821,466865,57,GPU-series-2,10.0,1.0,11,15894066.0,463.0,Spot,15894529.0


In [31]:
test_hp = hp_jobs.iloc[0]

print("Test HP job:")
print(f"  submit_time : {test_hp['submit_time']}")
print(f"  end_time    : {test_hp['end_time']}")
print(f"  duration    : {test_hp['duration']}")
print(f"  gpu_model   : {test_hp['gpu_model']}")

Test HP job:
  submit_time : 0.0
  end_time    : 2764799.0
  duration    : 2764799.0
  gpu_model   : A10


In [34]:
# For first HP job
overlapping=spot_jobs[
    (spot_jobs['submit_time']<test_hp['end_time'])& # Shows spot jobs are running before HP jobs finishes
    (spot_jobs['end_time']>test_hp['submit_time'])  # Shows spot jobs finish faster than when HP started
]

In [35]:
print(f"\nSpot jobs running at same time: {len(overlapping)}")
print(f"Total spot workers during this HP job: {overlapping['worker_num'].sum()}")


Spot jobs running at same time: 362
Total spot workers during this HP job: 1035


In [36]:
results = []

for idx, hp_job in hp_jobs.iterrows():
    overlapping_spot = spot_jobs[
        (spot_jobs['submit_time'] < hp_job['end_time']) &
        (spot_jobs['end_time'] > hp_job['submit_time'])
    ]
    
    # how many spot jobs overlapped
    concurrent_spot_jobs = len(overlapping_spot)
    
    # adding all their workers
    concurrent_spot_workers = overlapping_spot['worker_num'].sum()

    results.append({
        'job_name'              : hp_job['job_name'],
        'gpu_model'             : hp_job['gpu_model'],
        'worker_num'            : hp_job['worker_num'],
        'submit_time'           : hp_job['submit_time'],
        'duration'              : hp_job['duration'],
        'cpu_request'           : hp_job['cpu_request'],
        'gpu_request'           : hp_job['gpu_request'],
        'concurrent_spot_jobs'  : concurrent_spot_jobs,
        'concurrent_spot_workers': concurrent_spot_workers
    })

hp_features = pd.DataFrame(results)

print(f"Done. Shape: {hp_features.shape}")
print(hp_features.head())

Done. Shape: (415713, 9)
   job_name     gpu_model  worker_num  submit_time    duration  cpu_request  \
0    239255           A10           1          0.0   2764799.0         20.0   
1    253689           A10           1          0.0  15897599.0          8.0   
2    236907           A10           1          0.0   9857174.0          8.0   
3    253671  GPU-series-1           1          0.0   1727999.0          4.0   
4    236901           A10           1          0.0  15897599.0          8.0   

   gpu_request  concurrent_spot_jobs  concurrent_spot_workers  
0          1.0                   362                     1035  
1          1.0                 51153                   214154  
2          1.0                 22556                    94651  
3          1.0                     3                        3  
4          1.0                 51153                   214154  


In [37]:
hp_features['baseline_duration'] = hp_features.groupby(
    ['gpu_model', 'worker_num']
)['duration'].transform('median')

In [38]:
# Finding deviation from baseline duration
hp_features['deviation'] = (
    (hp_features['duration'] - hp_features['baseline_duration'])
    / hp_features['baseline_duration']
).clip(lower=0)

In [39]:
print("Deviation stats:")
print(hp_features['deviation'].describe())

# baseline for a few groups
print("\nSample baselines per GPU model + worker_num:")
print(hp_features.groupby(['gpu_model', 'worker_num'])['baseline_duration']
      .first()
      .reset_index()
      .head(10))

Deviation stats:
count    415713.000000
mean        345.741351
std        2252.605600
min           0.000000
25%           0.000000
50%           0.000000
75%           4.343902
max       38773.631707
Name: deviation, dtype: float64

Sample baselines per GPU model + worker_num:
        gpu_model  worker_num  baseline_duration
0             A10           1              410.0
1             A10           2             1369.0
2             A10           8            11347.5
3             A10          16             1427.0
4             A10          32             1260.0
5  A100-SXM4-80GB           1             1825.0
6  A100-SXM4-80GB           2             4085.5
7  A100-SXM4-80GB           3             3999.0
8  A100-SXM4-80GB           4             4051.5
9  A100-SXM4-80GB           5             5734.5


In [41]:
percentiles = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99, 0.999]

print("Deviation percentiles:")
for p in percentiles:
    val = hp_features['deviation'].quantile(p)
    print(f"  {int(p*100)}th percentile: {val:.4f}")

print("\nConcurrent spot workers percentiles:")
for p in percentiles:
    val = hp_features['concurrent_spot_workers'].quantile(p)
    print(f"  {int(p*100)}th percentile: {val:.1f}")

# how many jobs have ZERO deviation?
zero_dev = (hp_features['deviation'] == 0).sum()
print(f"\nJobs with zero deviation: {zero_dev} ({zero_dev/len(hp_features)*100:.1f}%)")

Deviation percentiles:
  50th percentile: 0.0000
  60th percentile: 0.8195
  70th percentile: 2.9364
  75th percentile: 4.3439
  80th percentile: 7.2732
  85th percentile: 12.6268
  90th percentile: 36.7287
  95th percentile: 631.1927
  99th percentile: 10957.1644
  99th percentile: 34228.5252

Concurrent spot workers percentiles:
  50th percentile: 119.0
  60th percentile: 209.2
  70th percentile: 320.0
  75th percentile: 387.0
  80th percentile: 468.0
  85th percentile: 599.0
  90th percentile: 935.0
  95th percentile: 6311.8
  99th percentile: 81463.5
  99th percentile: 214151.0

Jobs with zero deviation: 207946 (50.0%)


In [42]:
deviation_cap = hp_features['deviation'].quantile(0.95)
hp_features['deviation_capped'] = hp_features['deviation'].clip(upper=deviation_cap)

print(f"Deviation capped at: {deviation_cap:.4f}")

# deviation > 0.5 means the job ran 50% slower than its peers
# spot_workers > 387 means it faced above average spot load (75th percentile)
DEVIATION_THRESHOLD = 0.5
SPOT_WORKERS_THRESHOLD = 387

# 1 = high contention risk
# 0 = low risk
# both conditions must be true at the same time
# because a slow job with no spot load is not contention
# and high spot load with no slowdown means the HP job was fine
hp_features['contention_label'] = (
    (hp_features['deviation_capped'] > DEVIATION_THRESHOLD) &
    (hp_features['concurrent_spot_workers'] > SPOT_WORKERS_THRESHOLD)
).astype(int)

# check the label balance
label_counts = hp_features['contention_label'].value_counts()
label_pct = hp_features['contention_label'].value_counts(normalize=True) * 100

print("\nLabel distribution:")
print(f"  Label 0 (low risk) : {label_counts[0]} jobs ({label_pct[0]:.1f}%)")
print(f"  Label 1 (high risk): {label_counts[1]} jobs ({label_pct[1]:.1f}%)")

Deviation capped at: 631.1927

Label distribution:
  Label 0 (low risk) : 343657 jobs (82.7%)
  Label 1 (high risk): 72056 jobs (17.3%)


In [43]:
# total concurrent jobs = concurrent spot jobs + 1 (the HP job itself)
hp_features['total_concurrent_jobs'] = hp_features['concurrent_spot_jobs'] + 1

# ratio of spot jobs to total jobs running at that moment
# tells us how "spot heavy" the cluster was during this HP job
hp_features['spot_load_ratio'] = (
    hp_features['concurrent_spot_jobs'] / 
    hp_features['total_concurrent_jobs']
)

# arrival rate: how many jobs arrived in a 1-hour window before this HP job
# we define a 1-hour window = 3600 seconds
# for each HP job, count how many total jobs submitted in the hour before it
WINDOW = 3600  

# sorting all jobs by submit time first
all_jobs_sorted = job_df.sort_values('submit_time').reset_index(drop=True)

# for each HP job, counting jobs that arrived in the hour before it
# this measures how "busy" the cluster was arriving at that moment
arrival_rates = []

for _, hp_job in hp_features.iterrows():
    window_start = hp_job['submit_time'] - WINDOW
    window_end   = hp_job['submit_time']
    
    # counting all jobs (HP and Spot) that arrived in this window
    count = all_jobs_sorted[
        (all_jobs_sorted['submit_time'] >= window_start) &
        (all_jobs_sorted['submit_time'] < window_end)
    ].shape[0]
    
    arrival_rates.append(count)

hp_features['arrival_rate_1h'] = arrival_rates

print("New features added:")
print(hp_features[['job_name', 'spot_load_ratio', 
                    'arrival_rate_1h', 'contention_label']].head(10))

print("\nSpot load ratio stats:")
print(hp_features['spot_load_ratio'].describe())

print("\nArrival rate stats:")
print(hp_features['arrival_rate_1h'].describe())

New features added:
   job_name  spot_load_ratio  arrival_rate_1h  contention_label
0    239255         0.997245                0                 1
1    253689         0.999980                0                 1
2    236907         0.999956                0                 1
3    253671         0.750000                0                 0
4    236901         0.999980                0                 1
5    253670         0.750000                0                 0
6    253629         0.999977                0                 1
7    236897         0.750000                0                 0
8    377234         0.500000                0                 0
9    253611         0.999977                0                 1

Spot load ratio stats:
count    415713.000000
mean          0.797668
std           0.369678
min           0.000000
25%           0.937500
50%           0.971429
75%           0.987342
max           0.999980
Name: spot_load_ratio, dtype: float64

Arrival rate stats:
count    

In [44]:
final_df = hp_features[[
    'job_name',               
    'gpu_model',              
    'cpu_request',            
    'gpu_request',            
    'worker_num',             
    'concurrent_spot_jobs',   
    'concurrent_spot_workers',
    'spot_load_ratio',        # proportion of spot vs total
    'arrival_rate_1h',        # how busy was cluster in last hour
    'contention_label'        # our engineered label (0 or 1)
]].copy()

print("Missing values per column:")
print(final_df.isnull().sum())

print(f"\nFinal dataset shape: {final_df.shape}")

print("\nFinal label distribution:")
print(final_df['contention_label'].value_counts())

print("\nFirst 5 rows:")
print(final_df.head())

final_df.to_csv('hp_contention_dataset.csv', index=False)
print("\nSaved to hp_contention_dataset.csv")

Missing values per column:
job_name                   0
gpu_model                  0
cpu_request                0
gpu_request                0
worker_num                 0
concurrent_spot_jobs       0
concurrent_spot_workers    0
spot_load_ratio            0
arrival_rate_1h            0
contention_label           0
dtype: int64

Final dataset shape: (415713, 10)

Final label distribution:
contention_label
0    343657
1     72056
Name: count, dtype: int64

First 5 rows:
   job_name     gpu_model  cpu_request  gpu_request  worker_num  \
0    239255           A10         20.0          1.0           1   
1    253689           A10          8.0          1.0           1   
2    236907           A10          8.0          1.0           1   
3    253671  GPU-series-1          4.0          1.0           1   
4    236901           A10          8.0          1.0           1   

   concurrent_spot_jobs  concurrent_spot_workers  spot_load_ratio  \
0                   362                     1035      